In [ ]:
"""
Step 1: Update iowa_vendor serving layer (Parquet & Delta)
Deterministic enrichment with CSV dictionary, synthetic names, IsActive, State, City.
"""
from pathlib import Path
import shutil
import random
import duckdb
import pandas as pd
from deltalake import write_deltalake, DeltaTable

BASE_DIR = Path("_lakehouse/iowa_sales")
PARQUET_FILE = BASE_DIR / "layer_serving/parquet/iowa_vendor.snappy.parquet"
DELTA_DIR = BASE_DIR / "layer_serving/delta/iowa_vendor"
CSV_GLOB = str(BASE_DIR / "layer_drp/iowa_liquor_sales_2026_1263_rows_part_*.csv")

US_STATE_CITIES = {
    "Iowa": ["Des Moines", "Cedar Rapids", "Davenport", "Sioux City", "Iowa City"],
    "Kentucky": ["Louisville", "Lexington", "Frankfort", "Bowling Green", "Owensboro"],
    "Tennessee": ["Nashville", "Memphis", "Knoxville", "Chattanooga", "Clarksville"],
    "California": ["Los Angeles", "San Francisco", "San Diego", "San Jose", "Sacramento"],
    "New York": ["New York", "Buffalo", "Rochester", "Albany", "Syracuse"],
    "Texas": ["Austin", "Houston", "Dallas", "San Antonio", "Fort Worth"],
    "Illinois": ["Chicago", "Springfield", "Peoria", "Naperville", "Rockford"],
    "Pennsylvania": ["Philadelphia", "Pittsburgh", "Allentown", "Erie", "Harrisburg"],
    "Ohio": ["Columbus", "Cleveland", "Cincinnati", "Toledo", "Akron"],
    "Missouri": ["Kansas City", "St. Louis", "Springfield", "Columbia", "Independence"],
    "Indiana": ["Indianapolis", "Fort Wayne", "Evansville", "South Bend", "Carmel"],
    "Wisconsin": ["Milwaukee", "Madison", "Green Bay", "Kenosha", "Appleton"],
    "Colorado": ["Denver", "Colorado Springs", "Aurora", "Fort Collins", "Boulder"],
    "Washington": ["Seattle", "Spokane", "Tacoma", "Vancouver", "Bellevue"],
    "Florida": ["Miami", "Orlando", "Tampa", "Jacksonville", "St. Petersburg"],
    "Georgia": ["Atlanta", "Savannah", "Augusta", "Athens", "Macon"],
    "North Carolina": ["Charlotte", "Raleigh", "Greensboro", "Durham", "Winston-Salem"],
    "Michigan": ["Detroit", "Grand Rapids", "Warren", "Ann Arbor", "Lansing"],
    "Oregon": ["Portland", "Eugene", "Salem", "Bend", "Hillsboro"],
    "Minnesota": ["Minneapolis", "St. Paul", "Rochester", "Duluth", "Bloomington"]
}
STATES_LIST = sorted(list(US_STATE_CITIES.keys()))

PREFIXES = [
    "Heritage", "Pinnacle", "Summit", "Oakwood", "Grand River", "Heartland",
    "Copper Craft", "Prairie Ridge", "Blue Valley", "Silver Creek", "Redwood",
    "Frontier", "Crown & Barrel", "Great Lakes", "Cascade", "Ironwood",
    "Keystone", "Golden State", "Lone Star", "Ambassador", "Timberline",
    "Highland", "Old Colony", "Meridian", "Apex"
]
CORES = [
    "Spirits", "Distilling", "Beverage Group", "Vintners", "Distributors",
    "Craft Liquors", "Winery & Spirits", "Cellars", "Trading Co.",
    "Import & Distribution", "Brands", "Distillers"
]
SUFFIXES = ["LLC", "Inc.", "Co.", "Group", "Holdings"]

def format_title_case(name: str) -> str:
    """Format company names to proper Title Case preserving acronyms."""
    words = name.split()
    formatted = []
    for w in words:
        wl = w.lower()
        if wl in ["usa", "llc", "inc", "inc.", "corp", "corp.", "dno"]:
            formatted.append(
                wl.upper()
                .replace("INC", "Inc.")
                .replace("CORP", "Corp.")
                .replace("LLC", "LLC")
                .replace("USA", "USA")
            )
        else:
            formatted.append(w.capitalize())
    return " ".join(formatted)

def main():
    print("=== Rozpoczynam Krok 1: Wzbogacenie iowa_vendor ===")
    
    bak_parquet = PARQUET_FILE.with_suffix(".parquet.bak")
    if not bak_parquet.exists() and PARQUET_FILE.exists():
        shutil.copy2(PARQUET_FILE, bak_parquet)
        print(f"[Backup] Zapisano kopię Parquet: {bak_parquet}")

    bak_delta = DELTA_DIR.parent / f"{DELTA_DIR.name}_bak"
    if not bak_delta.exists() and DELTA_DIR.exists():
        shutil.copytree(DELTA_DIR, bak_delta)
        print(f"[Backup] Zapisano kopię Delta: {bak_delta}")

    con = duckdb.connect()
    existing_df = con.execute(f'SELECT VendorID, Inserted FROM "{PARQUET_FILE}"').df()
    
    csv_df = con.execute(f"""
        SELECT DISTINCT 
            TRY_CAST(vendor_number AS BIGINT) as VendorID, 
            TRIM(vendor_name) as raw_name
        FROM read_csv_auto('{CSV_GLOB}', union_by_name=true)
        WHERE TRY_CAST(vendor_number AS BIGINT) IS NOT NULL AND vendor_name IS NOT NULL
    """).df()
    csv_dict = dict(zip(csv_df["VendorID"], csv_df["raw_name"]))

    print(f"Liczba wierszy w bazie: {len(existing_df)}")
    print(f"Liczba unikalnych vendorów w CSV: {len(csv_dict)}")

    records = []
    used_names = set()

    for _, row in existing_df.iterrows():
        vid = int(row["VendorID"])
        inserted = row["Inserted"]
        
        # Deterministyczny generator na podstawie VendorID
        rnd = random.Random(vid + 42)
        state = rnd.choice(STATES_LIST)
        city = rnd.choice(US_STATE_CITIES[state])

        if vid in csv_dict:
            is_active = True
            vname = format_title_case(csv_dict[vid])
        else:
            is_active = False
            # Deterministyczna unikalna nazwa syntetyczna
            p = rnd.choice(PREFIXES)
            c = rnd.choice(CORES)
            s = rnd.choice(SUFFIXES)
            candidate = f"{p} {c} {s}"
            while candidate in used_names:
                candidate = f"{p} {c} #{vid} {s}"
            vname = candidate

        used_names.add(vname)
        records.append({
            "VendorID": vid,
            "VendorName": vname,
            "IsActive": bool(is_active),
            "State": state,
            "City": city,
            "Inserted": inserted
        })

    enriched_df = pd.DataFrame(records)

    assert len(enriched_df) == 531, f"Oczekiwano 531 wierszy, otrzymano {len(enriched_df)}"
    assert enriched_df["VendorName"].isna().sum() == 0, "Wykryto wartości NULL w VendorName!"
    assert not enriched_df["VendorName"].str.startswith("$").any(), "Wykryto pozostałości cen '$'!"

    active_cnt = enriched_df["IsActive"].sum()
    inactive_cnt = len(enriched_df) - active_cnt
    print(f"Aktywnych (z CSV): {active_cnt}, Nieaktywnych (syntetycznych): {inactive_cnt}")

    enriched_df.to_parquet(PARQUET_FILE, index=False)
    print(f"[Sukces] Zapisano zaktualizowany Parquet: {PARQUET_FILE}")

    write_deltalake(DELTA_DIR, enriched_df, mode="overwrite", schema_mode="overwrite")
    print(f"[Sukces] Zapisano zaktualizowany Delta Lake: {DELTA_DIR}")

    print("\n--- Próbka danych (Active): ---")
    print(enriched_df[enriched_df["IsActive"]].head(5)[["VendorID", "VendorName", "IsActive", "State", "City"]])
    print("\n--- Próbka danych (Inactive): ---")
    print(enriched_df[~enriched_df["IsActive"]].head(5)[["VendorID", "VendorName", "IsActive", "State", "City"]])

if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
from IPython.display import display
from deltalake import DeltaTable
import duckdb

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 1000)

df_parquet = duckdb.query('SELECT * FROM "_lakehouse/iowa_sales/layer_serving/parquet/iowa_vendor.snappy.parquet" LIMIT 100').df()
display(df_parquet)

df_delta = DeltaTable("_lakehouse/iowa_sales/layer_serving/delta/iowa_vendor").to_pandas()
display(df_delta.head(100))